# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(url)

# Show dataset metadata name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Examine all record sets and their fields, referencing by '@id'
if not dataset.metadata.record_sets:
    print("WARNING: No record sets detected in Croissant metadata.\nmlcroissant may parse records from CSVs based on files instead. Listing available files:")
    for i, fileobj in enumerate(dataset.metadata.file_objects):
        print(f"{i+1}. File '@id': {fileobj.id}, url: {fileobj.content_url}, name: {fileobj.name}")
    # Identify which file object(s) contain records
    # In the FAIR^2 schema, often a records CSV is referenced by a FileObject in encoding/distribution.
        
else:
    for rs in dataset.metadata.record_sets:
        print(f"RecordSet: @id={rs.id}, name={rs.name}")
        for field in rs.fields:
            print(f"  Field: @id={field.id}, name={field.name}, data type={field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# In this dataset, records are typically stored in major record set CSV; extract all available records into DataFrames.
# For FAIR^2, use FileObject @id for extraction, since record_sets are sometimes empty in schema.

# List to collect file object '@id's to extract as record sets
file_record_ids = [fo.id for fo in dataset.metadata.file_objects]

print("Record sets or file objects available for extraction:")
for file_id in file_record_ids:
    print(f"- @id: {file_id}")

dataframes = {}
# Try to extract from each file object
for file_id in file_record_ids:
    try:
        recs = list(dataset.records(record_set=file_id))
        if recs:
            df = pd.DataFrame(recs)
            dataframes[file_id] = df
            print(f"Loaded {len(df)} records for '@id': {file_id}, columns: {df.columns.tolist()}")
        else:
            print(f"No records found for '@id': {file_id}")
    except Exception as e:
        print(f"Could not load records from '@id': {file_id}: {e}")

# For demonstration, use the first available DataFrame
main_record_set_id = next(iter(dataframes))
print(f"\nPreviewing top records from '@id': {main_record_set_id}")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Example: Explore numeric fields (e.g., MW [kDa] - Molecular Weight, Peptides, Coverage)
df = dataframes[main_record_set_id]
print("Available columns:", df.columns.tolist())

# Try to auto-detect a numeric field
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi' and col.lower() not in ['id', 'accession']]
if not numeric_candidates:
    # Try to convert some known columns to numeric
    for col in df.columns:
        if any(sub in col.lower() for sub in ['mw', 'peptide', 'coverage', 'abundance', 'score', 'intensity']):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                pass
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi']

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.columns[0]  # fallback

print(f"Using numeric field '@id' (or column name): {numeric_field}")

# Set demonstration threshold for filtering
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a key protein annotation, e.g. 'Description' or 'Gene'
possible_groups = [col for col in df.columns if col.lower() in ['description', 'gene', 'accession']]
if possible_groups:
    group_field = possible_groups[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=40, kde=True, color="skyblue")
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If normalized, show a boxplot grouped by group_field
if 'group_field' in locals() and group_field in filtered_df.columns:
    top_groups = filtered_df[group_field].value_counts().nlargest(6).index
    plt.figure(figsize=(10,6))
    sns.boxplot(
        x=group_field, 
        y=f"{numeric_field}_normalized", 
        data=filtered_df[filtered_df[group_field].isin(top_groups)]
    )
    plt.title(f"Normalized {numeric_field} by {group_field} (top 6 groups)")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides mass spectrometry-based protein information, with properties such as abundance and molecular descriptors for extracellular vesicles from human mast cells.
- We loaded and inspected the primary records CSV (referenced by its file object `@id`), examining numeric data like molecular weight and peptide counts.
- Simple EDA steps included filtering, normalizing, and grouping; visualizations highlighted data distributions.
- For more detailed protein annotation or case study, consult the Croissant metadata and relevant file object `@id`s for advanced filtering and analysis.
